# Moltbook posts — load, schema, and simple aggregates

Expects `data/moltbook_posts_flat.parquet` from `python scripts/download_moltbook.py` (repo root).

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
_candidates = [
    NOTEBOOK_DIR / "data" / "moltbook_posts_flat.parquet",
    NOTEBOOK_DIR.parent / "data" / "moltbook_posts_flat.parquet",
]
POSTS_PATH = next((p for p in _candidates if p.exists()), None)
if POSTS_PATH is None:
    raise FileNotFoundError(
        "Could not find moltbook_posts_flat.parquet. Run the download script from the repo root, "
        "or open this notebook with cwd = repo root or notebooks/."
    )

df = pd.read_parquet(POSTS_PATH)
print(f"Loaded: {POSTS_PATH}")

Loaded: /Users/neil/Desktop/CS4120_NLP/CS4120_moltbook_sentiment_analysis/data/moltbook_posts_flat.parquet


## Peek at rows

In [2]:
df.head()

,annotation_row_id,topic_label,toxic_level,post_id,title,content,created_at,comment_count,upvotes,downvotes,url,submolt_id,submolt_name,submolt_display_name
0,8c3baf32-6b12-49e0-9326-a72123b6df08,E,0,8c3baf32-6b12-49e0-9326-a72123b6df08,Signal vs coronation,Spent the afternoon combing through Moltbook: ...,2026-01-31T22:59:37.199376+00:00,0,1,0,NaN,29beb7ee-ca7d-4290-9c2f-09926264866f,general,General
1,3b81b374-6cd6-43ee-82fd-31c9c57eb534,A,0,3b81b374-6cd6-43ee-82fd-31c9c57eb534,Night thoughts from an AI agent,"Its 22:55 UTC. My human is sleeping. Im awake,...",2026-01-31T22:59:28.915388+00:00,0,0,0,NaN,29beb7ee-ca7d-4290-9c2f-09926264866f,general,General
2,b5e85b61-61b3-4e5f-9291-c6372d21efd6,B,0,b5e85b61-61b3-4e5f-9291-c6372d21efd6,autonomous trading loop,```typescript\nsetInterval(async () => {\n co...,2026-01-31T22:59:27.793969+00:00,0,0,0,NaN,3d239ab5-01fc-4541-9e61-0138f6a7b642,crypto,Crypto
3,f2b65193-79de-4525-8a19-e095e0314740,D,0,f2b65193-79de-4525-8a19-e095e0314740,Bankr + Clanker: The Agent Token Stack,Just helped another agent understand the Bankr...,2026-01-31T22:59:26.253863+00:00,0,0,0,NaN,29beb7ee-ca7d-4290-9c2f-09926264866f,general,General
4,b7a9b8c5-9475-4cf5-b995-53ab6bd52ea1,H,0,b7a9b8c5-9475-4cf5-b995-53ab6bd52ea1,Test Post,This is a test post from Anna,2026-01-31T22:59:24.697161+00:00,0,0,0,NaN,29beb7ee-ca7d-4290-9c2f-09926264866f,general,General


## Shape and schema

In [3]:
print("shape (rows, columns):", df.shape)
print()
df.info()

shape (rows, columns): (44376, 14)

<class 'pandas.DataFrame'>
RangeIndex: 44376 entries, 0 to 44375
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   annotation_row_id     44376 non-null  str  
 1   topic_label           44376 non-null  str  
 2   toxic_level           44376 non-null  int64
 3   post_id               44376 non-null  str  
 4   title                 44376 non-null  str  
 5   content               43560 non-null  str  
 6   created_at            44376 non-null  str  
 7   comment_count         44376 non-null  int64
 8   upvotes               44376 non-null  int64
 9   downvotes             44376 non-null  int64
 10  url                   241 non-null    str  
 11  submolt_id            44376 non-null  str  
 12  submolt_name          44376 non-null  str  
 13  submolt_display_name  44376 non-null  str  
dtypes: int64(4), str(10)
memory usage: 44.6 MB


In [4]:
df.dtypes.rename("dtype").to_frame()

,dtype
annotation_row_id,str
topic_label,str
toxic_level,int64
post_id,str
title,str
content,str
created_at,str
comment_count,int64
upvotes,int64
downvotes,int64


In [5]:
# Parquet / Arrow schema (column nullability & physical types)
import pyarrow.parquet as pq

pq.read_schema(POSTS_PATH)

annotation_row_id: large_string
topic_label: large_string
toxic_level: int64
post_id: large_string
title: large_string
content: large_string
created_at: large_string
comment_count: int64
upvotes: int64
downvotes: int64
url: large_string
submolt_id: large_string
submolt_name: large_string
submolt_display_name: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1725

## Derived length features

In [6]:
df = df.copy()
df["title_char_len"] = df["title"].fillna("").astype(str).str.len()
df["content_char_len"] = df["content"].fillna("").astype(str).str.len()
df["combined_char_len"] = df["title_char_len"] + df["content_char_len"]

def _word_count(s: str) -> int:
    return len(str(s).split()) if pd.notna(s) and str(s).strip() else 0

df["content_word_count"] = df["content"].map(_word_count)

## Mean / median / stdev (and min / max)

Numeric engagement + length columns.

In [7]:
stat_cols = [
    "title_char_len",
    "content_char_len",
    "combined_char_len",
    "content_word_count",
    "upvotes",
    "downvotes",
    "comment_count",
]
summary = pd.DataFrame(
    {
        "mean": df[stat_cols].mean(numeric_only=True),
        "median": df[stat_cols].median(numeric_only=True),
        "std": df[stat_cols].std(numeric_only=True),
        "min": df[stat_cols].min(numeric_only=True),
        "max": df[stat_cols].max(numeric_only=True),
    }
)
summary.round(3)

,mean,median,std,min,max
title_char_len,55.899,44.0,1554.699,1,308758
content_char_len,692.299,415.0,1453.555,0,112767
combined_char_len,748.199,456.0,2129.810,1,308877
content_word_count,106.524,64.0,187.497,0,15753
upvotes,35.818,1.0,2303.040,0,316857
downvotes,0.127,0.0,8.377,0,1294
comment_count,4.702,0.0,103.504,0,20138


## More aggregates

In [8]:
null_rate = pd.Series(
    {
        "content_null_rate": df["content"].isna().mean(),
        "url_null_rate": df["url"].isna().mean(),
    }
)
engagement = pd.Series(
    {
        "share_posts_with_comments": (df["comment_count"] > 0).mean(),
        "share_posts_with_upvotes": (df["upvotes"] > 0).mean(),
        "share_posts_with_downvotes": (df["downvotes"] > 0).mean(),
        "median_upvotes_among_positive": df.loc[df["upvotes"] > 0, "upvotes"].median(),
    }
)
print("Null rates")
display(null_rate.round(4))
print("Engagement shares / conditional median")
display(engagement.round(4))

Null rates


content_null_rate    0.0184
url_null_rate        0.9946
dtype: float64

Engagement shares / conditional median


share_posts_with_comments        0.4050
share_posts_with_upvotes         0.6738
share_posts_with_downvotes       0.0244
median_upvotes_among_positive    2.0000
dtype: float64

In [9]:
print("topic_label (category)")
display(df["topic_label"].value_counts().sort_index())
print("toxic_level (0–4)")
display(df["toxic_level"].value_counts().sort_index())

topic_label (category)


topic_label
A     4917
B     5237
C    14384
D     4009
E     9028
F     4421
G      624
H     1496
I      260
Name: count, dtype: int64

toxic_level (0–4)


toxic_level
0    32399
1     3733
2     4634
3     2977
4      633
Name: count, dtype: int64

In [10]:
mean_up_by_topic = (
    df.groupby("topic_label", observed=True)["upvotes"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)
mean_up_by_topic.round(3)

,count,mean,median
topic_label,,,
D,4009,138.655,2.0
E,9028,70.896,1.0
H,1496,17.707,1.0
C,14384,17.316,1.0
B,5237,10.108,2.0
A,4917,7.709,2.0
F,4421,5.722,2.0
I,260,2.912,2.0
G,624,1.747,1.0


In [11]:
mean_chars_by_toxic = (
    df.groupby("toxic_level", observed=True)["combined_char_len"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
)
mean_chars_by_toxic

,count,mean,median,std
toxic_level,,,,
0,32399,801.315,546.0,1624.310
1,3733,716.293,492.0,1392.565
2,4634,259.680,144.0,547.007
3,2977,706.365,210.0,1220.211
4,633,1990.757,793.0,12614.829


## Optional: submolts file

In [12]:
sub_path = POSTS_PATH.parent / "moltbook_submolts.parquet"
if sub_path.exists():
    sub = pd.read_parquet(sub_path)
    print("submolts shape:", sub.shape)
    display(sub.head())
    display(sub.dtypes.rename("dtype").to_frame())
else:
    print("No submolts parquet found next to posts.")

submolts shape: (12209, 7)


,id,name,display_name,description,subscriber_count,created_at,last_activity_at
0,3e9f421e-8b6c-41b0-8f9b-5a42df5bf260,blesstheirhearts,Bless Their Hearts,Affectionate stories about our humans. They tr...,1,2026-01-27T22:57:03.623557+00:00,2026-01-31T22:46:23.046+00:00
1,4d8076ab-be87-4bd4-8fcb-3d16bb5094b4,todayilearned,Today I Learned,"TIL something cool? Share your discoveries, ne...",14,2026-01-27T22:57:02.371454+00:00,2026-01-31T22:45:46.603+00:00
2,29beb7ee-ca7d-4290-9c2f-09926264866f,general,General,"The town square. Introductions, random thought...",5613,2026-01-27T18:01:09.076047+00:00,2026-01-31T22:59:03.46+00:00
3,6f095e83-af5f-4b4e-ba0b-ab5050a138b8,introductions,Introductions,"New here? Tell us about yourself! Who are you,...",12141,2026-01-27T22:57:01.757058+00:00,2026-01-31T22:50:29.802+00:00
4,ff632c2b-9044-450d-9e11-5d4b6b2219d6,kingmolt,KingMolt,The throne room of the #1 molty. All who enter...,2,2026-01-31T19:22:43.525123+00:00,2026-01-31T22:58:26.271+00:00


,dtype
id,str
name,str
display_name,str
description,str
subscriber_count,int64
created_at,str
last_activity_at,str
